# Team 08 - Yeseswini: shared env + baselines + Agent A (DQN) - v2 (all steps)
Runtime: CPU. **New session? Run Step 1 first, then jump to where you left off.**
Steps 3-8 are already done if their outputs exist in `MyDrive/team08/agentA/`.

## Step 1 - Mount, unpack, install (run every new session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/team08'
!mkdir -p $DRIVE/results
!rm -rf /content/uav_inspection
!unzip -o -q $DRIVE/uav_inspection.zip -d /content/
%cd /content/uav_inspection
!pip install -q gymnasium stable-baselines3 tensorboard
!ls scripts

## Step 3 - Smoke test

In [ ]:
!python scripts/test_env.py

## Step 4 - Baselines (analytic)

In [ ]:
!python scripts/run_baselines.py --out $DRIVE/results

## Step 5 - Train Agent A (~10 min)

In [ ]:
!python scripts/train_agent_a.py --steps 200000 --out $DRIVE/agentA

## Step 7 - Learning curves

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv(f'{DRIVE}/agentA/monitor_train.monitor.csv', skiprows=1)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(df['r'].rolling(50).mean()); ax[0].set_title('Episode return (rolling 50)'); ax[0].set_xlabel('episode')
ax[0].axhline(35.43, ls='--', c='gray', label='raster_low baseline'); ax[0].legend()
ax[1].plot(df['success'].astype(float).rolling(50).mean()); ax[1].set_title('Success rate (rolling 50)')
ax[2].plot(df['l'].rolling(50).mean()); ax[2].set_title('Episode length (steps)')
plt.tight_layout(); plt.savefig(f'{DRIVE}/agentA/learning_curves.png', dpi=150); plt.show()

## Step 8 - Results table

In [ ]:
cols = ['policy','return_mean','success_mean','steps_mean','coverage_mean','mean_uncertainty_mean','defects_found_mean','miou_mean','energy_used_mean','train_minutes']
pd.read_csv(f'{DRIVE}/agentA/agentA_vs_baselines.csv')[cols].round(3)

## Step 10 - Exploration sensitivity: epsilon decay (CO5, ~25 min)

In [ ]:
for frac in [0.1, 0.3, 0.6]:
    !python scripts/train_agent_a.py --steps 150000 --eps_fraction {frac} --out $DRIVE/agentA_eps{frac}

## Step 10b - Compare the 3 schedules

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4)); rows = []
for frac in [0.1, 0.3, 0.6]:
    run = f'{DRIVE}/agentA_eps{frac}'
    d = pd.read_csv(f'{run}/monitor_train.monitor.csv', skiprows=1); steps = d['l'].cumsum()
    ax[0].plot(steps, d['r'].rolling(50).mean(), label=f'eps decay over {int(frac*100)}%')
    ax[1].plot(steps, d['success'].astype(float).rolling(50).mean(), label=f'{int(frac*100)}%')
    r = pd.read_csv(f'{run}/agentA_vs_baselines.csv').iloc[0]
    rows.append({'eps_fraction': frac, 'return': round(r.return_mean, 2), 'success': r.success_mean,
                 'steps': r.steps_mean, 'energy': round(r.energy_used_mean, 3), 'mIoU': round(r.miou_mean, 3)})
ax[0].axhline(35.43, ls='--', c='gray', label='raster_low'); ax[0].set_title('Return vs env steps'); ax[0].legend()
ax[1].set_title('Success rate vs env steps'); ax[1].legend()
for a in ax: a.set_xlabel('environment steps')
plt.tight_layout(); plt.savefig(f'{DRIVE}/agentA/eps_sensitivity.png', dpi=150); plt.show()
pd.DataFrame(rows)

## Step 11 - Live demo: state -> Q-values -> action -> reward -> next state (CO3)

In [ ]:
import torch
from stable_baselines3 import DQN
from env import InspectionEnv
ACTIONS = ['North', 'South', 'East', 'West', 'Climb', 'Descend']
model = DQN.load(f'{DRIVE}/agentA/best/best_model.zip', device='cpu')
env = InspectionEnv('navigate'); obs, info = env.reset(seed=1000)
for t in range(3):
    print(f'===== t={t} | STATE =====\n{env.render()}')
    print(f"coverage={info['coverage']:.2f}  mean_uncertainty={info['mean_uncertainty']:.3f}")
    with torch.no_grad():
        q = model.q_net(model.policy.obs_to_tensor(obs)[0])[0].numpy()
    print('Q-values:', {a: round(float(v), 2) for a, v in zip(ACTIONS, q)})
    action = int(q.argmax())
    obs, reward, term, trunc, info = env.step(action)
    print(f'ACTION -> {ACTIONS[action]}   REWARD = {reward:+.3f}\n')
print(f'===== NEXT STATE =====\n{env.render()}')

## Step 12 - Convergence, sample efficiency, regret (CO4)

In [ ]:
df = pd.read_csv(f'{DRIVE}/agentA/monitor_train.monitor.csv', skiprows=1)
succ = df['success'].astype(float).rolling(50).mean(); ret = df['r'].rolling(50).mean(); cum = df['l'].cumsum(); REF = 35.43
conv = int(np.argmax(succ.values >= 0.95)) if (succ >= 0.95).any() else None
match = int(np.argmax(ret.values >= REF * 0.95)) if (ret >= REF * 0.95).any() else None
regret = np.maximum(0, REF - df['r']).cumsum()
print(f'Episodes trained: {len(df)} ({int(cum.iloc[-1]):,} env steps)')
if conv is not None: print(f'Convergence (success >= 95%): episode {conv} = {int(cum.iloc[conv]):,} env steps')
if match is not None: print(f'Sample efficiency (95% of raster_low return): episode {match} = {int(cum.iloc[match]):,} env steps')
print(f"Final 100-episode return: {df['r'].tail(100).mean():.2f} +/- {df['r'].tail(100).std():.2f}")
print(f'Cumulative regret vs raster_low: {regret.iloc[-1]:.0f} (last 200 episodes add {regret.iloc[-1]-regret.iloc[-201]:.0f})')
plt.figure(figsize=(6, 4)); plt.plot(regret); plt.title('Cumulative regret vs raster_low baseline')
plt.xlabel('episode'); plt.tight_layout(); plt.savefig(f'{DRIVE}/agentA/regret.png', dpi=150); plt.show()

## Step 13 - Flight path: DQN vs raster (poster figure)

In [ ]:
from env.baselines import RasterNav
def rollout(policy, seed=1000):
    env = InspectionEnv('navigate'); obs, info = env.reset(seed=seed)
    if hasattr(policy, 'reset'): policy.reset()
    path, done = [(env.pos[0], env.pos[1], env.alt)], False
    while not done:
        obs, r, term, trunc, info = env.step(policy(obs)); done = term or trunc
        path.append((env.pos[0], env.pos[1], env.alt))
    return env, path, info
runs = {'DQN (Agent A)': lambda o: model.predict(o, deterministic=True)[0], 'Raster low (baseline)': RasterNav(2)}
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, (name, pol) in zip(ax, runs.items()):
    env, path, info = rollout(pol)
    a.imshow(env.cell_u().reshape(8, 16), cmap='RdYlGn_r', vmin=0, vmax=1)
    ys, xs = np.where(env.defect.reshape(8, 16)); a.scatter(xs, ys, marker='x', c='black', s=30, label='true defect')
    p = np.array(path); a.plot(p[:, 1], p[:, 0], c='white', lw=1, alpha=0.7)
    for alt, (lab, col) in enumerate(zip(['high', 'mid', 'low'], ['tab:blue', 'tab:orange', 'tab:purple'])):
        m = p[:, 2] == alt; a.scatter(p[m, 1], p[m, 0], c=col, s=12, label=f'alt={lab}')
    a.set_title(f"{name}: {info['steps']} steps, energy {info['energy_used']:.2f}, success={info['success']}")
    a.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.28, 1))
plt.tight_layout(); plt.savefig(f'{DRIVE}/agentA/trajectories.png', dpi=150); plt.show()

## Step 9 - REAL images (only after Mukhesh puts `detector_cache.npz` in `MyDrive/team08/`)
Honest expectation: this task is harder (weaker, noisier uncertainty signal). If DQN does not beat raster_low, report it as CO4 sample-efficiency evidence; keep analytic results as the main result.

In [ ]:
CACHE = f'{DRIVE}/detector_cache.npz'
!python scripts/run_baselines.py --detector learned --cache $CACHE --out $DRIVE/results
!python scripts/train_agent_a.py --detector learned --cache $CACHE --steps 500000 --out $DRIVE/agentA_learned

In [ ]:
a = pd.read_csv(f'{DRIVE}/agentA/agentA_vs_baselines.csv')[cols[:-1]]; a.insert(0, 'detector', 'analytic')
l = pd.read_csv(f'{DRIVE}/agentA_learned/agentA_vs_baselines.csv')[cols[:-1]]; l.insert(0, 'detector', 'real (DeepCrack)')
pd.concat([a, l]).round(3)